# 03 — Fonds-CO₂-Fußabdruck & ESG-Clustering mit MLflow

Dieses Notebook baut ein klassisches Data-Science-Projekt **in die Lakehouse-Architektur** ein:

1. **Feature-Extraktion** aus dem Lakehouse: pro Fonds die gewichteten Scope-1/2-Emissionen seiner Holdings (Trino → Iceberg `raw`-Layer)
2. **Clustering** der Fonds in ESG-/Karbon-Profile (KMeans, k-Sweep)
3. **MLflow-Tracking**: jeder Lauf wird als Run mit Parametern, Metriken, Modell und Plot geloggt
4. **Model Registry**: das beste Modell wird registriert

Alles wird in der MLflow-UI reported: **http://localhost:5555**

> Backend-Store = PostgreSQL (DB `mlflow`), Artifact-Store = MinIO (Bucket `mlflow`) — dieselben Services, die das Lakehouse ohnehin hat.

In [4]:
import os
import matplotlib.pyplot as plt
import pandas as pd
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from trino.dbapi import connect

# MLFLOW_TRACKING_URI ist im jupyter-Container gesetzt (http://mlflow:5000)
mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", "http://mlflow:5000"))
mlflow.set_experiment("fonds-co2-fussabdruck")
print("Tracking-URI:", mlflow.get_tracking_uri())

Tracking-URI: http://mlflow:5000


## 1. Feature-Extraktion aus dem Lakehouse

Pro Fonds (letzter Stichtag) verbinden wir die Positionen mit den Emissionsdaten der gehaltenen Unternehmen und gewichten mit `weight_pct` (summiert je Fonds auf 100 %):

- `w_scope1`, `w_scope2` — gewichtete Scope-1/2-Emissionen (Financed-Emissions-Proxy)
- `pct_energy`, `pct_materials` — Exposure in CO₂-intensiven Sektoren
- `n_holdings` — Anzahl Positionen

In [5]:
EMISSIONS_YEAR = 2023

FEATURE_SQL = f"""
WITH latest AS (SELECT max(position_date) AS d FROM raw.fund_positions),
     emis AS (
        SELECT isin, sector, scope_1_tco2e, scope_2_market_tco2e
        FROM raw.nzdpu_emissions
        WHERE reporting_year = {EMISSIONS_YEAR}
     )
SELECT
    m.fund_name,
    m.fund_type,
    count(*)                                              AS n_holdings,
    sum(p.weight_pct / 100.0 * e.scope_1_tco2e)           AS w_scope1,
    sum(p.weight_pct / 100.0 * e.scope_2_market_tco2e)    AS w_scope2,
    sum(p.weight_pct) FILTER (WHERE e.sector = 'Energy')      AS pct_energy,
    sum(p.weight_pct) FILTER (WHERE e.sector = 'Materials')   AS pct_materials
FROM raw.fund_positions p
JOIN latest l   ON p.position_date = l.d
JOIN emis e     ON p.holding_isin = e.isin
JOIN raw.fund_master m ON p.fund_isin = m.fund_isin
GROUP BY m.fund_name, m.fund_type
ORDER BY w_scope1 DESC
"""

conn = connect(host="trino", port=8080, user="mlflow", catalog="nessie", schema="raw")
df = pd.read_sql(FEATURE_SQL, conn)
df[["pct_energy", "pct_materials"]] = df[["pct_energy", "pct_materials"]].fillna(0.0)
df

/tmp/ipykernel_168/2539486675.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(FEATURE_SQL, conn)


,fund_name,fund_type,n_holdings,w_scope1,w_scope2,pct_energy,pct_materials
0,Deka-Nachhaltigkeit Renten CF,Rentenfonds,17,1.202876e+07,373591.821648,21.6121,11.1390
1,Deka-ESG MSCI World Climate Paris CF,Indexfonds,19,9.542620e+06,370438.596446,17.4642,5.1224
2,DekaFonds CF,Aktienfonds,16,9.162753e+06,323910.218673,15.7850,5.2247
3,Deka-Wandelanleihen CF,Mischfonds,16,7.619489e+06,373434.711929,11.0982,9.4886
4,Deka-EuropaSelect CF,Aktienfonds,19,6.917670e+06,377100.872397,12.3768,3.2006
5,Deka-Klimawandel & Biodiversitaet CF,Thematischer Fonds,18,6.806246e+06,356004.689854,10.2671,8.7476
6,Deka-GlobalChampions CF,Aktienfonds,18,6.463591e+06,402781.114200,9.9541,9.8518
7,Deka-Nachhaltigkeit Aktien CF,Aktienfonds,13,5.313677e+06,337796.494746,9.9103,0.0000
8,Deka-Europa Aktien Spezial CF,Aktienfonds,20,5.187429e+06,288110.074514,7.6529,6.4712
9,Deka-Basisstrategie Aktien CF,Aktienfonds,13,4.797052e+06,301224.387896,4.9420,11.1350


## 2. Standardisierung

KMeans ist distanzbasiert — die Features müssen auf vergleichbare Skalen gebracht werden.

In [ ]:
FEATURES = ["n_holdings", "w_scope1", "w_scope2", "pct_energy", "pct_materials"]
X = df[FEATURES].to_numpy(dtype="float64")
X_scaled = StandardScaler().fit_transform(X)
X_scaled.shape

## 3. k-Sweep mit MLflow-Tracking

Für jedes `k` starten wir einen MLflow-Run und loggen Parameter, Silhouette-Score, Inertia, das Modell und einen Cluster-Plot. Danach in der UI (http://localhost:5555) vergleichbar.

In [ ]:
results = []
for k in range(2, 7):
    with mlflow.start_run(run_name=f"kmeans_k{k}"):
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = km.fit_predict(X_scaled)
        sil = silhouette_score(X_scaled, labels)

        mlflow.log_param("algorithm", "KMeans")
        mlflow.log_param("k", k)
        mlflow.log_param("features", ",".join(FEATURES))
        mlflow.log_param("emissions_year", EMISSIONS_YEAR)
        mlflow.log_metric("silhouette", sil)
        mlflow.log_metric("inertia", km.inertia_)

        signature = infer_signature(X_scaled, labels)
        mlflow.sklearn.log_model(km, artifact_path="model", signature=signature)

        fig, ax = plt.subplots(figsize=(7, 5))
        sc = ax.scatter(df["w_scope1"], df["pct_energy"], c=labels, cmap="viridis", s=120)
        for _, row in df.iterrows():
            ax.annotate(row["fund_name"][:18], (row["w_scope1"], row["pct_energy"]), fontsize=7, alpha=0.7)
        ax.set_xlabel("Gewichtete Scope-1-Emissionen")
        ax.set_ylabel("Energy-Exposure (%)")
        ax.set_title(f"Fonds-Cluster (k={k}, Silhouette={sil:.3f})")
        fig.colorbar(sc, label="Cluster")
        fig.tight_layout()
        plot_path = f"/tmp/clusters_k{k}.png"
        fig.savefig(plot_path, dpi=110)
        mlflow.log_artifact(plot_path, artifact_path="plots")

        results.append((k, sil, mlflow.active_run().info.run_id))
        print(f"k={k}: silhouette={sil:.3f}, inertia={km.inertia_:.1f}")
        plt.show()

## 4. Bestes Modell → Model Registry

Wir wählen das `k` mit dem höchsten Silhouette-Score und registrieren das Modell. Danach ist es in der UI unter **Models** versioniert.

In [ ]:
best_k, best_sil, best_run = max(results, key=lambda r: r[1])
print(f"Bestes Modell: k={best_k} (silhouette={best_sil:.3f})")

mlflow.register_model(f"runs:/{best_run}/model", "fonds-esg-clustering")

## 5. Gelabelter Fonds-Report

Zum Abschluss labeln wir alle Fonds mit ihrem Cluster und loggen den Report als Artefakt. Cluster 0 = CO₂-intensiver, Cluster 1 = emissionsärmer (bei k=2).

In [ ]:
with mlflow.start_run(run_name=f"final_labeled_k{best_k}"):
    km = KMeans(n_clusters=best_k, random_state=42, n_init=10)
    df["cluster"] = km.fit_predict(X_scaled)
    mlflow.log_param("k", best_k)
    mlflow.log_metric("silhouette", best_sil)
    df.to_csv("/tmp/fonds_co2_report.csv", index=False)
    mlflow.log_artifact("/tmp/fonds_co2_report.csv", artifact_path="report")

df[["fund_name", "fund_type", "w_scope1", "pct_energy", "cluster"]].sort_values("cluster")

---
✅ **Fertig.** Öffne jetzt die MLflow-UI: **http://localhost:5555**

- **Experiments → fonds-co2-fussabdruck**: alle Runs, Silhouette/Inertia je `k` vergleichen
- **Runs → Artifacts**: Cluster-Plots und den Fonds-Report ansehen
- **Models → fonds-esg-clustering**: die registrierte Modellversion (Stage auf *Staging*/*Production* setzen)